# Project 637 – Resume Classification

# Business objective -
The document classification solution should significantly reduce the manual human effort in the HRM. It should achieve a higher level of accuracy and automation with minimal human intervention

# PHASE A — PROJECT SETUP & DISCOVERY

In [5]:
# ================================
# PHASE A1: ENVIRONMENT SETUP
# ================================

# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [6]:
# Project root directory
PROJECT_ROOT = "/Resume_Classification_Project"
RAW_DATA_DIR = f"{PROJECT_ROOT}/data/raw"
EXTRACTED_TEXT_DIR = f"{PROJECT_ROOT}/data/extracted_text"
METADATA_DIR = f"{PROJECT_ROOT}/data/metadata"
CLEAN_DATA_DIR = f"{PROJECT_ROOT}/data/cleaned"

import os
# Create folder structure
for path in [
    PROJECT_ROOT,
    RAW_DATA_DIR,
    EXTRACTED_TEXT_DIR,
    METADATA_DIR,
    CLEAN_DATA_DIR
]:
    os.makedirs(path, exist_ok=True)

print("Project folder structure created successfully.")

Project folder structure created successfully.


In [7]:
# ================================
# PHASE A2: UNZIP DATASET
# ================================

import zipfile

zip_path = "P637 Dataset.zip"
extract_path = RAW_DATA_DIR

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset unzipped successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'P637 Dataset.zip'

In [ ]:
# ================================
# PHASE A3: DATA INSPECTION
# ================================

from collections import Counter

# Corrected resume_root path
resume_root = os.path.join(RAW_DATA_DIR, "Resumes")

file_extensions = []
label_distribution = Counter()

for root, dirs, files in os.walk(resume_root):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        file_extensions.append(ext)
        label = os.path.basename(root)
        label_distribution[label] += 1

print("File format distribution:")
print(Counter(file_extensions))

print("\nLabel distribution:")
for label, count in label_distribution.items():
    print(f"{label}: {count}")

In [ ]:
# ================================
# PHASE A4: LABEL TAXONOMY
# ================================

import pandas as pd

taxonomy = []

for label in label_distribution.keys():
    taxonomy.append({
        "raw_label": label,
        "canonical_label": label.strip().lower().replace(" ", "_")
    })

label_df = pd.DataFrame(taxonomy)
label_df.to_csv(f"{METADATA_DIR}/label_taxonomy.csv", index=False)

label_df.head()

In [ ]:
label_df.loc[label_df['raw_label'] == 'Resumes', 'raw_label'] = 'React Developer'
label_df.loc[label_df['canonical_label'] == 'resumes', 'canonical_label'] = 'react_developer'
label_df.loc[label_df['raw_label']== 'SQL Developer Lightning insight', 'raw_label'] = 'SQL Developer'
label_df.loc[label_df['canonical_label'] == 'sql_developer_lightning_insight', 'canonical_label'] = 'sql_developer'


# Save the updated label_df to CSV
label_df.to_csv(f"{METADATA_DIR}/label_taxonomy.csv", index=False)

label_df.head()

# PHASE B — TEXT EXTRACTION

In [ ]:
# ================================
# PHASE B1: INSTALL DEPENDENCIES
# ================================

# System-level dependencies
!apt-get update
!apt-get install -y antiword poppler-utils tesseract-ocr

# Python libraries
!pip install docx2txt pdfplumber pytesseract pillow

In [ ]:
# DOCX extractor (tables + text)
import docx2txt

def extract_text_docx_safe(path):
    try:
        text = docx2txt.process(path)
        return text.strip()
    except Exception as e:
        return ""
# DOC extractor (legacy Word files)
import subprocess

def extract_text_doc_safe(path):
    try:
        result = subprocess.run(
            ["antiword", path],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True
        )
        return result.stdout.strip()
    except Exception:
        return ""
# PDF text extractor (non-scanned PDFs)
import pdfplumber

def extract_text_pdf_safe(path):
    text = ""
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception:
        pass
    return text.strip()
# OCR extractor (scanned PDFs)
import pytesseract
from PIL import Image
import tempfile
import os

def extract_text_pdf_ocr(path):
    text = ""
    try:
        with tempfile.TemporaryDirectory() as temp_dir:
            subprocess.run(
                ["pdftoppm", path, f"{temp_dir}/page", "-png"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            for img_file in sorted(os.listdir(temp_dir)):
                img_path = os.path.join(temp_dir, img_file)
                text += pytesseract.image_to_string(Image.open(img_path))
    except Exception:
        pass
    return text.strip()

In [ ]:
# Master extraction function (FINAL, USE THIS)
import os

def extract_resume_text(path):
    ext = os.path.splitext(path)[1].lower()
    text = ""

    if ext == ".docx":
        text = extract_text_docx_safe(path)

    elif ext == ".doc":
        text = extract_text_doc_safe(path)

    elif ext == ".pdf":
        text = extract_text_pdf_safe(path)
        # OCR fallback for scanned PDFs
        if len(text) < 50:
            text = extract_text_pdf_ocr(path)

    else:
        # Unsupported format
        return ""

    return text.strip()

In [ ]:
# Re-run extraction safely (with failure logging)
import pandas as pd

records = []
failed_files = []

for root, dirs, files in os.walk(resume_root):
    for file in files:
        file_path = os.path.join(root, file)
        label = os.path.basename(root)

        text = extract_resume_text(file_path)

        if len(text) < 50:
            failed_files.append(file_path)

        records.append({
            "filename": file,
            "label": label,
            "raw_text": text
        })

raw_text_df = pd.DataFrame(records)

# Correct the 'Resumes' label to 'React Developer' in the main DataFrame
raw_text_df['label'] = raw_text_df['label'].replace('Resumes', 'React Developer')
raw_text_df['label'] = raw_text_df['label'].replace('SQL Developer Lightning insight', 'SQL Developer')

raw_text_df.to_csv(f"{EXTRACTED_TEXT_DIR}/raw_resume_text_fixed.csv", index=False)

pd.DataFrame(
    failed_files, columns=["file_path"]
).to_csv(
    f"{PROJECT_ROOT}/data/metadata/failed_extractions.csv",
    index=False
)

print(f"Total resumes processed: {len(records)}")
print(f"Failed extractions: {len(failed_files)}")

In [ ]:
raw_text_df["text_len"] = raw_text_df["raw_text"].apply(len)

raw_text_df.sort_values("text_len").sample(10)

# PHASE C — DATA CLEANING & PREPROCESSING

In [ ]:
# Load raw extracted text
df = pd.read_csv(f"{EXTRACTED_TEXT_DIR}/raw_resume_text_fixed.csv")

In [ ]:
df = df[df["raw_text"].notna()]
df = df[df["raw_text"].str.len() > 50].reset_index(drop=True)

print(df.shape)

In [ ]:
# Regex-based text cleaning (resume-specific)
import re
def clean_text_regex(text):
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)       # email
    text = re.sub(r'\+?\d[\d\s\-()]{8,}', ' ', text)  # phone
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["clean_text"] = df["raw_text"].apply(clean_text_regex)

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy

# Load spaCy + POS tagging
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

In [ ]:
# Tokenization + POS filtering (ONLY NOUNS & VERBS)
def extract_nouns_verbs(text):
    doc = nlp(text)
    nouns = []
    verbs = []

    for token in doc:
        if token.is_stop or len(token.text) < 3:
            continue
        if token.pos_ == "NOUN":
            nouns.append(token.lemma_)
        elif token.pos_ == "VERB":
            verbs.append(token.lemma_)

    return nouns, verbs

df["nouns"], df["verbs"] = zip(*df["clean_text"].apply(extract_nouns_verbs))

In [ ]:
df.head()

In [ ]:
# Count nouns & verbs per resume
df["noun_count"] = df["nouns"].apply(len)
df["verb_count"] = df["verbs"].apply(len)

df[["label", "noun_count", "verb_count"]].describe()

In [ ]:
CLEAN_DIR = f"{PROJECT_ROOT}/data/linguistic"
import os
os.makedirs(CLEAN_DIR, exist_ok=True)

df.to_csv(f"{CLEAN_DIR}/resumes_linguistic_cleaned.csv", index=False)

print("✅ Phase C completed")

In [ ]:
# Define robust section patterns

SECTION_PATTERNS = {
    "skills": r"(skills|technical skills|core competencies|technologies)",
    "experience": r"(experience|work experience|professional experience|employment history)",
    "projects": r"(projects|project experience|academic projects)",
    "education": r"(education|academic background|qualification)"
}

In [ ]:
# Section extraction function

def extract_sections_from_text(text):
    text = text.lower()
    sections = {
        "skills": "",
        "experience": "",
        "projects": "",
        "education": ""
    }

    matches = []
    for section, pattern in SECTION_PATTERNS.items():
        for match in re.finditer(pattern, text):
            matches.append((match.start(), section))

    if not matches:
        return sections

    matches.sort(key=lambda x: x[0])

    for i, (start_idx, section) in enumerate(matches):
        end_idx = matches[i + 1][0] if i + 1 < len(matches) else len(text)
        sections[section] += text[start_idx:end_idx]

    return sections

In [ ]:
# Apply section extraction to dataframe
df["sections"] = df["clean_text"].apply(extract_sections_from_text)

# Sanity check
df[["sections"]].head(3)

In [ ]:
# Define a REALISTIC skill vocabulary (expandable)
SKILL_VOCAB = {
    # Programming
    "python": ["python"],
    "java": ["java"],
    "c++": ["c++", "cpp"],
    "javascript": ["javascript", "js"],
    "typescript": ["typescript", "ts"],

    # Web / Frameworks
    "react": ["react", "reactjs"],
    "node": ["node", "nodejs"],
    "angular": ["angular"],
    "django": ["django"],
    "flask": ["flask"],

    # Databases
    "sql": ["sql", "mysql", "postgresql", "oracle"],
    "mongodb": ["mongodb", "mongo"],

    # BI / Analytics
    "powerbi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "excel": ["excel"],

    # ML / DS
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning", "dl"],
    "nlp": ["nlp", "natural language processing"],

    # Cloud / Tools
    "aws": ["aws"],
    "azure": ["azure"],
    "git": ["git"],
    "docker": ["docker"],

    # ERP
    "peoplesoft": ["peoplesoft"],
    "workday": ["workday"]
}

In [ ]:
# Section-aware skill extraction
def extract_skills_from_sections(sections):
    combined_text = " ".join([
        sections.get("skills", ""),
        sections.get("experience", ""),
        sections.get("projects", "")
    ])

    extracted = set()
    combined_text = combined_text.lower()

    for canonical, variants in SKILL_VOCAB.items():
        for v in variants:
            if re.search(rf"\b{re.escape(v)}\b", combined_text):
                extracted.add(canonical)

    return list(extracted)

In [ ]:
df["skills"] = df["sections"].apply(extract_skills_from_sections)

# Sanity check
df[["label", "skills"]].sample(10)

In [ ]:
df["num_skills"] = df["skills"].apply(len)

no_skill_df = df[df["num_skills"] == 0]

print("Resumes with no detected skills:", len(no_skill_df))

# Log for audit
no_skill_df.to_csv(
    f"{PROJECT_ROOT}/data/metadata/resumes_without_skills.csv",
    index=False
)

# Keep valid resumes
df = df[df["num_skills"] > 0].reset_index(drop=True)

In [ ]:
FINAL_DATA_DIR = f"{PROJECT_ROOT}/data/final"
import os
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

df.to_csv(
    f"{FINAL_DATA_DIR}/resume_dataset_with_sections_and_skills.csv",
    index=False
)

print("✅ Sections and skills successfully created.")

In [ ]:
df.head()

In [ ]:
# ============================
# FINAL DATAFRAME FOR EDA & MODELING
# ============================

df_final = df[
    [
        "label",
        "nouns",
        "verbs",
        "noun_count",
        "verb_count",
        "sections",
        "skills"
    ]
].copy()

df_final.head()

# PHASE D — PERFECT, MEANINGFUL EDA

In [ ]:
# Global Top 20 NOUNS & VERBS (frequency)
from collections import Counter

all_nouns = [n for sub in df_final["nouns"] for n in sub]
all_verbs = [v for sub in df_final["verbs"] for v in sub]

top_nouns = Counter(all_nouns).most_common(20)
top_verbs = Counter(all_verbs).most_common(20)

Global Top 20 Nouns — Insight

Dominant nouns reflect technical domains and system components, indicating resumes emphasize tools, platforms, and technologies rather than soft skills.

Global Top 20 Verbs — Insight

High-frequency action verbs show resumes are achievement-oriented, useful for identifying hands-on versus theoretical profiles.

In [ ]:
# Bar plots — Top 20 Nouns & Verbs
import matplotlib.pyplot as plt
import pandas as pd

noun_df = pd.DataFrame(top_nouns, columns=["noun", "count"])
verb_df = pd.DataFrame(top_verbs, columns=["verb", "count"])

plt.figure(figsize=(10,4))
plt.bar(noun_df["noun"], noun_df["count"])
plt.xticks(rotation=90)
plt.title("Top 20 Most Frequent Nouns in Resumes")
plt.show()

plt.figure(figsize=(10,4))
plt.bar(verb_df["verb"], verb_df["count"])
plt.xticks(rotation=90)
plt.title("Top 20 Most Frequent Verbs in Resumes")
plt.show()

Bar Plot (Top Nouns & Verbs) — Insight

Clear frequency drop after top terms suggests few core technical concepts dominate most resumes, enabling effective dimensionality reduction.

In [ ]:
# WordCloud — Nouns & Verbs (semantic insight)
from wordcloud import WordCloud

noun_wc = WordCloud(width=800, height=400, background_color="white").generate(
    " ".join(all_nouns)
)

verb_wc = WordCloud(width=800, height=400, background_color="white").generate(
    " ".join(all_verbs)
)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.imshow(noun_wc)
plt.axis("off")
plt.title("Noun WordCloud")

plt.subplot(1,2,2)
plt.imshow(verb_wc)
plt.axis("off")
plt.title("Verb WordCloud")
plt.show()

WordCloud (Nouns) — Insight

Noun clusters highlight technology stacks and business systems, validating domain-specific vocabulary alignment.

WordCloud (Verbs) — Insight

Verb emphasis on develop, implement, analyze confirms resumes are task-centric, useful for role suitability scoring.

In [ ]:
# Number of profiles (resumes) per label
label_counts = df_final["label"].value_counts()
label_counts

Number of Profiles per Label — Insight

Class distribution reveals potential imbalance, indicating need for stratified sampling or class weighting during modeling.

In [ ]:
# Donut chart — profile distribution (HR-friendly)
fig = plt.figure(figsize=(8,8))
plt.pie(
    label_counts.values,
    labels=label_counts.index,
    autopct="%1.0f%%",
    startangle=140,
    wedgeprops=dict(width=0.4),
    explode = (0.01, 0.01, 0.01, 0.01)
)
plt.title("Resume Distribution by Profile")
plt.show()

Donut Chart (Profile Distribution) — Insight

Visual clarity helps HR quickly assess talent pool composition, aiding resource allocation and hiring strategy.

In [ ]:
# Top 20 most used words (CountVectorizer on nouns+verbs)
from sklearn.feature_extraction.text import CountVectorizer

df_final["nv_text"] = df_final.apply(
    lambda x: " ".join(x["nouns"] + x["verbs"]),
    axis=1
)

vectorizer = CountVectorizer(max_features=20)
X = vectorizer.fit_transform(df_final["nv_text"])

word_counts = X.sum(axis=0).A1

cv_df = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "count": word_counts
}).sort_values(by="count", ascending=False)

cv_df

Top 20 Words (CountVectorizer) — Insight

High-frequency tokens largely overlap with technical skills, reinforcing the importance of explicit skill extraction.

In [ ]:
# Resume length analysis (linguistic depth)
df_final["resume_length"] = df_final["noun_count"] + df_final["verb_count"]

df_final["resume_length"].hist(bins=40, figsize=(8,4))
plt.title("Resume Linguistic Length Distribution")
plt.xlabel("Nouns + Verbs Count")
plt.ylabel("Frequency")
plt.show()

Resume Length Distribution — Insight

Wide variation in linguistic length indicates resume verbosity inconsistency, justifying normalization during feature engineering.

In [ ]:
# Noun vs Verb usage (resume writing style insight)
plt.figure(figsize=(6,4))
plt.scatter(
    df_final["noun_count"],
    df_final["verb_count"],
    alpha=0.5
)
plt.xlabel("Noun Count")
plt.ylabel("Verb Count")
plt.title("Noun vs Verb Distribution in Resumes")
plt.show()

Noun vs Verb Usage Scatter — Insight

Balanced noun-verb usage suggests well-structured resumes, while outliers may indicate poor quality or non-technical profiles.

In [ ]:
# Skill count distribution (resume richness)
df_final["skill_count"] = df_final["skills"].apply(len)

df_final["skill_count"].hist(bins=20, figsize=(8,4))
plt.title("Skill Count Distribution per Resume")
plt.xlabel("Number of Skills")
plt.ylabel("Frequency")
plt.show()

Skill Count Distribution — Insight

Most resumes list a moderate number of skills, confirming skill-based filtering is reliable without excessive sparsity.

In [ ]:
# Top skills per profile (CORE INSIGHT)
from collections import defaultdict

skill_profile_map = defaultdict(list)

for _, row in df_final.iterrows():
    for skill in row["skills"]:
        skill_profile_map[row["label"]].append(skill)

top_skills_per_profile = {
    label: Counter(skills).most_common(10)
    for label, skills in skill_profile_map.items()
}

top_skills_per_profile

Top Skills per Profile — Insight

Distinct skill clusters per label confirm strong discriminative power of technical skills for classification.

In [ ]:
# Skill overlap heatmap (confusion risk)
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

labels = df_final["label"].unique()
all_skills = sorted({s for skills in df_final["skills"] for s in skills})
skill_index = {s: i for i, s in enumerate(all_skills)}

skill_vectors = {}

for label in labels:
    vectors = []
    for skills in df_final[df_final["label"] == label]["skills"]:
        vec = np.zeros(len(all_skills))
        for s in skills:
            vec[skill_index[s]] = 1
        vectors.append(vec)
    skill_vectors[label] = np.mean(vectors, axis=0)

similarity_matrix = np.zeros((len(labels), len(labels)))

for i, l1 in enumerate(labels):
    for j, l2 in enumerate(labels):
        similarity_matrix[i, j] = cosine_similarity(
            skill_vectors[l1].reshape(1, -1),
            skill_vectors[l2].reshape(1, -1)
        )[0][0]

pd.DataFrame(similarity_matrix, index=labels, columns=labels)

Skill Overlap (Confusion Risk) — Insight

Higher similarity between certain roles signals natural confusion zones, guiding targeted feature enrichment or model tuning.

DATA PREPROCESSING

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

label_encoder = LabelEncoder()


In [ ]:
df_final

In [ ]:
df_final.describe()

In [ ]:
TEXT_COLS = ["nouns", "verbs", "skills", "sections", "nv_text"]

for col in TEXT_COLS:
    df_final[col] = df_final[col].astype(str)

In [ ]:
df_final = df_final[df_final["nv_text"].str.len() > 20].reset_index(drop=True)

In [ ]:
df_final

In [ ]:
TEXT_FEATURES = ["skills", "nv_text"]
NUM_FEATURES = ["noun_count", "verb_count", "resume_length", "skill_count"]

X = df_final[TEXT_FEATURES + NUM_FEATURES]


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

# Use original text labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_final["label"])

print("Classes:", label_encoder.classes_)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape,X_test.shape

In [ ]:
TEXT_FEATURES = ["skills", "nv_text"]
NUM_FEATURES = ["noun_count", "verb_count", "resume_length", "skill_count"]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

In [ ]:
column_transformer = ColumnTransformer(
    transformers=[
        ("skills_tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1,2),
            min_df=3,
            max_df=0.9,
            max_features=3000
        ), "skills"),

        ("nv_tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1,2),
            min_df=3,
            max_df=0.9,
            max_features=3000
        ), "nv_text"),

        ("num", MinMaxScaler(), NUM_FEATURES)
    ]
)

In [ ]:
# This should work without errors
X_train_transformed = column_transformer.fit_transform(X_train)

print("Transformed train shape:", X_train_transformed.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        # n_jobs=-1,
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        # n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
("num", MinMaxScaler(), NUM_FEATURES)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, accuracy_score

results = []

for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", column_transformer),
            ("classifier", model)
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Support": len(y_test)
    })


In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by="F1-score",
    ascending=False
)

In [ ]:
results_df

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for model_name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", column_transformer),
            ("classifier", model)
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=skf,
        scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"]
    )

    cv_results.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision_macro"].mean(),
        "CV Recall": scores["test_recall_macro"].mean(),
        "CV F1-score": scores["test_f1_macro"].mean(),
        "CV Support": len(y)
    })


In [ ]:
cv_results_df = pd.DataFrame(cv_results).sort_values(
    by="CV F1-score",
    ascending=False
)

cv_results_df

Several models achieved perfect scores on the test split due to the small test size. To ensure robustness, I evaluated all models using stratified 5-fold cross-validation and selected the best model based on F1-score.

In [ ]:
final_model_name = "Linear SVM"

final_model = Pipeline(
    steps=[
        ("preprocessor", column_transformer),
        ("classifier", LinearSVC(
            class_weight="balanced",
            random_state=42
        ))
    ]
)

In [ ]:
final_model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = final_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Downloading the pickle file

In [ ]:
import pickle

# Save trained pipeline
pickle.dump(final_model, open("resume_classifier.pkl", "wb"))

# Save fitted label encoder
pickle.dump(label_encoder, open("label_encoder.pkl", "wb"))

